In [1]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

# treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    use_sud=False,
    matrix_type="coverage",
    # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
        excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own"]
)

connected to port: 53951


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(corpus.feature_matrix)

In [11]:
import numpy as np
results = []
for idx, lex_unit in corpus._idx2lexunit.items():
    current_category = lex_unit[1]
    closest_neighbours = np.argsort(similarity_matrix[idx])[:-21:-1]
    closest_neighbours = [i for i in closest_neighbours if i != idx][:20]
    for neighbour_idx in closest_neighbours:
        neighbour_category = corpus._idx2lexunit[neighbour_idx][1]
        if neighbour_category != current_category:
            diff = np.abs(corpus.feature_matrix[idx] - corpus.feature_matrix[neighbour_idx])
            closest_features_idx= np.argsort(diff)[:-6:-1]
            closest_features = [corpus._idx2feature[i] for i in closest_features_idx]
                
            results.append({
                "lex_unit": lex_unit,
                "neighbour_lex_unit": corpus._idx2lexunit[neighbour_idx],
                "closest_features": closest_features,
                
            })

for r in results:
    print(r)   

{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Factor', 'PROPN'), 'closest_features': ['node:X:child:rel_shallow=nummod', 'node:X:child:rel_shallow=flat', 'node:X:prev:upos=ADP', 'node:X:parent:Number=Sing', 'node:X:child:rel_shallow=punct']}
{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Q', 'PROPN'), 'closest_features': ['node:X:child:rel_shallow=flat', 'node:X:child:rel_shallow=nummod', 'node:X:parent:Number=Sing', 'node:X:child:rel_shallow=punct', 'node:X:child:upos=PUNCT']}
{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Wave', 'PROPN'), 'closest_features': ['node:X:child:rel_shallow=flat', 'node:X:child:rel_shallow=nummod', 'node:X:parent:upos=NOUN', 'node:X:parent:position=after', 'node:X:parent:Number=Sing']}
{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Figure', 'PROPN'), 'closest_features': ['node:X:child:rel_shallow=flat', 'node:X:child:rel_shallow=nummod', 'node:X:next:NumType=Frac', 'node:X:child:NumType=Frac', 'node:X:child:NumType=Card']}
{'lex_unit': ('$